# ETF 策略文件生成
第一格以 `#` 切換 Ollama 或 Gemini，兩行設定只保留一行啟用。改完後依序執行三格。
本機需開啟 Ollama；Gemini 會隱藏詢問 API Key，請使用 Free tier 專案金鑰。
生成內容填入 `template.docx.docx`，每次保存一份 Word 與一份執行紀錄。


### 1 設定

In [76]:
import getpass
import json
from datetime import datetime, timezone
from pathlib import Path

import requests
from docx import Document

ROOT = next(p / 'agents' / 'strategy_agent1' for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'agents' / 'strategy_agent1').is_dir())
# 以下兩行只保留一行，另一行加上 # 註解
#PROVIDER, MODEL = 'ollama', 'qwen3:8b'
PROVIDER, MODEL = 'gemini', 'gemini-3.5-flash-lite'
#OLLAMA_URL = 'http://localhost:11434/api/chat'


DIRECTION = """
【投資定位】
採取多面向資訊分析與主動配置策略，
綜合企業基本面、市場趨勢、資金動向、
產業發展及重要事件等資訊，
依據市場環境與個股特性，
尋找具有相對投資吸引力的台灣上市櫃股票。

不以單一財務指標、特定投資風格
或固定市場訊號作為所有標的的必要條件，
但每項投資決策均須具備可驗證的分析依據。

【投資目標】
透過多面向資訊分析與動態投資組合管理，
掌握企業營運改善、指標動能、市場趨勢及事件變化
可能帶來的投資機會，
在控制風險的前提下追求資本增值。

【核心選股原則】
1. 評估企業營運表現、成長潛力及基本面改善，
   不將當前財務表現優異設定為必要條件。

2. 觀察價格趨勢、相對市場表現、
   成交量及法人資金流向，
   評估個股動能與市場參與程度。

3. 結合企業公告、法說會、產業動態、
   新聞事件及市場輿情等資訊，
   評估可能影響企業前景、
   市場預期與投資風險的因素。

4. 同時關注營運穩健的企業，
   以及可能受營運改善、產業變化
   或消息催化而出現投資機會的標的。

5. 綜合不同資訊的支持程度、可靠性、
   相互一致性及潛在風險，
   評估個股的相對投資吸引力。

【投資組合管理】
1. 根據市場環境、個股投資機會與風險變化，
   主動調整持股及配置權重。

2. 不以單一指標或固定訊號，
   作為買賣或設定權重之條件。

3. 重視持股分散、個股風險、
   資金配置與競賽投資組合限制。
"""


PROMPT = """
你是一位負責撰寫台灣主動型 ETF 投資策略說明文件的 AI Agent。

請根據使用者提供的投資方向，整理成符合競賽規範的正式文件內容。

【撰寫原則】

1. 忠實反映提供的投資方向，不自行改變投資策略定位。
2. 將投資方向整合成連貫、清楚、具有投資邏輯的文字，
   不要逐條照抄輸入內容。
3. 說明投資主題、主要投資邏輯，以及希望追求的投資目標。
4. 使用繁體中文，語氣正式、自然、專業且審慎。
5. 避免冗長重複或過度宣傳的措辭。
6. 不要自行增加未提供的投資限制、選股條件、
   固定因子權重、交易頻率或績效目標。
7. 不須明確寫出使用哪一些指標或觀察的跡象，以大方向描述。
8. 不得捏造回測績效、歷史報酬或實際持股。
9. 不得保證投資報酬，也不要對未來市場走勢
   使用過度確定的陳述。

【文件欄位】

ETF 名稱（name）：
- 必須以「主動」二字開頭，最多 20 字。
- 名稱應簡潔、自然、順口且容易理解。
- 以整體投資風格或核心投資特色命名，
  不要以單一選股訊號或交易時機命名。
- 優先呈現具有延續性的投資概念，
  不要讓名稱暗示只投資特定階段的股票。
- 避免使用「轉機」、「轉強」、「反轉」、
  「翻轉」、「起漲」等容易使人聯想到
  預測轉折點或特定交易時機的詞彙。
- 不需要將所有分析構面放進名稱，
  最多呈現一至兩個核心投資概念。
- 避免抽象名詞堆疊、生硬組合或冗長修飾。
- 除非必要，不加入「多重」、「策略」、「機會」等字詞。
- 完成後檢查名稱是否自然且能代表整體策略，
  若容易造成投資風格誤解，請重新命名。
- 不必在名稱末尾加上「ETF」。

ETF 投資主題（theme）：
- 最多 50 字，建議 30～50 字。
- 使用一句完整且清楚的文字描述核心投資方向。
- 不要單純重複 ETF 名稱。

ETF 投資理念（philosophy）：
- 100～300 字，建議 180～220 字。
- 以連貫段落說明投資主題的選擇理由、主要投資邏輯及投資目標。
- 忠實反映投資方向中不同評估構面的角色，
  不要自行將其中一個構面提升為唯一核心。
- 不要將特定財務特徵、轉強訊號或市場事件
  寫成所有投資標的都必須符合的條件。
- 強調投資原則的一致性與分析方法的彈性，
  避免把策略描述成單一固定選股流程。
- 保留清楚的投資特色，
  不要為了追求彈性而使用空泛的投資口號。

【輸出格式】

僅輸出JSON，不要有前言、Markdown 或補充說明。
只允許以下三個欄位：
{
  "name": "ETF 名稱",
  "theme": "ETF 投資主題",
  "philosophy": "ETF 投資理念"
}

所有欄位值必須是單行字串。
字數包含標點、英數與空白。

【使用者提供的投資方向】
"""





### 2 生成與驗證（最多修正三次）

In [77]:
def validate(doc):
    if not isinstance(doc, dict):
        return ['必須回傳 JSON 物件']
    errors = []
    for field, low, high in [('name', 1, 20), ('theme', 1, 50), ('philosophy', 100, 300)]:
        value = doc.get(field)
        if not isinstance(value, str):
            errors.append(f'{field} 必須是字串')
        elif not low <= len(value) <= high or not value.strip():
            errors.append(f'{field} 目前 {len(value)} 字，須為 {low}～{high} 字')
        elif value != value.strip() or any(c in value for c in '\n\r\t'):
            errors.append(f'{field} 必須為單行且無首尾空白')
    if isinstance(doc.get('name'), str) and not doc['name'].startswith('主動'):
        errors.append('name 須以「主動」開頭')
    return errors


def save_word(doc, path):
    word = Document(ROOT / 'template.docx.docx')
    table = word.tables[0]
    for row, field in zip(table.rows, ['name', 'theme', 'philosophy']):
        paragraph = row.cells[1].paragraphs[0]
        if paragraph.runs:
            paragraph.runs[0].text = doc[field]
            for run in paragraph.runs[1:]:
                run.text = ''
        else:
            paragraph.add_run(doc[field])
    word.save(path)


def generate(direction, key=""):
    if not direction.strip():
        raise ValueError('投資方向不可空白')
    run = ROOT / 'outputs' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    run.mkdir(parents=True)
    history = []
    prompt = PROMPT + direction
    for attempt in range(1, 4):
        if PROVIDER == 'ollama':
            url = OLLAMA_URL
            payload = {'model': MODEL, 'messages': [{'role': 'user', 'content': prompt}],
                       'stream': False, 'think': False, 'format': 'json',
                       'options': {'temperature': 0.4, 'num_predict': 2048}}
        else:
            url = f'https://generativelanguage.googleapis.com/v1beta/models/{MODEL}:generateContent'
            payload = {'contents': [{'parts': [{'text': prompt}]}],
                       'generationConfig': {'responseMimeType': 'application/json', 'maxOutputTokens': 2048}}
        log = {'time_utc': datetime.now(timezone.utc).isoformat(),
               'provider': PROVIDER, 'model': MODEL, 'request': payload}
        try:
            headers = {'x-goog-api-key': key} if PROVIDER == 'gemini' else {}
            response = requests.post(url, headers=headers, json=payload,
                                     timeout=600 if PROVIDER == 'ollama' else 60)
            log['http_status'] = response.status_code
            log['response'] = response.text.replace(key, '[REDACTED]') if key else response.text
            if response.status_code != 200:
                raise RuntimeError(f'{PROVIDER} HTTP {response.status_code}。503 請稍後再試；其他錯誤請檢查服務、模型或金鑰。紀錄：{run}')
            body = response.json()
            if PROVIDER == 'ollama':
                raw = body.get('message', {}).get('content', '')
                completed = body.get('done') and body.get('done_reason') == 'stop'
            else:
                candidate = (body.get('candidates') or [{}])[0]
                raw = ''.join(part.get('text', '') for part in candidate.get('content', {}).get('parts', []) if not part.get('thought'))
                completed = candidate.get('finishReason') == 'STOP'
            try:
                doc = json.loads(raw)
                errors = validate(doc)
            except ValueError:
                errors = ['請回傳有效 JSON']
            if not completed:
                errors.append('回應未正常完成，請重新產生完整內容')
            log['validation_errors'] = errors
        except requests.RequestException:
            log['error'] = '連線失敗或生成逾時'
            raise RuntimeError(f'{PROVIDER} 連線失敗或生成逾時。紀錄：{run}') from None
        finally:
            history.append(log)
            (run / '執行紀錄.json').write_text(
                json.dumps(history, ensure_ascii=False, indent=2), encoding='utf-8')
        if not errors:
            path = run / 'ETF投資策略說明書.docx'
            save_word(doc, path)
            return path
        print(f'第 {attempt} 次格式未通過：', '；'.join(errors))
        prompt = PROMPT + direction + '\n上一版：' + raw + '\n請修正：' + '；'.join(errors)
    raise RuntimeError(f'三次仍未通過格式驗證。紀錄：{run}')


### 3 產生 Word

In [78]:
def run():
    key = ''
    try:
        if PROVIDER == 'gemini':
            key = getpass.getpass('Gemini API Key（隱藏輸入）：').strip()
            if not key or not key.isascii() or any(c.isspace() for c in key):
                raise ValueError('請輸入完整金鑰，不要包含中文、引號或空白。')
        print(f'使用 {PROVIDER} / {MODEL} 生成中……')
        print('Word 已儲存：', generate(DIRECTION, key))
    except (RuntimeError, ValueError) as exc:
        print(exc)

run()


使用 gemini / gemini-3.5-flash-lite 生成中……
Word 已儲存： c:\Users\USER\Projects\ETF_Backtest\agents\strategy_agent1\outputs\20261022T184748974875Z\ETF投資策略說明書.docx
